# 02 · Broad basketball feature research

**3,106 candidates; training-only screening; matched temporal ablations.** This is a deliberately broad hypothesis search, not a claim that every generated column is useful. Every displayed result is loaded from a completed, checksum-verified run of the code below. Review mode reads published evidence; train mode actually rebuilds and fits notebook 02.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from march_mania.notebook_support import table, style
from march_mania.publication.workflow import evidence, execution_mode, feature_stage, require_recorded_source
from march_mania.advanced_features import candidate_blocks
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file()
MODE = execution_mode()  # Set "train" to build or resume the current feature experiment.
style()
if MODE == "train":
    feature_stage(ROOT)
RESULTS, RECORD = evidence(ROOT, "feature_store")
require_recorded_source(ROOT, "feature_store", RECORD)
SUMMARY = RECORD["summary"]
assert SUMMARY["feature_count"] == len(candidate_blocks()["full"])
assert SUMMARY["status"] == "completed"
display(Markdown(f"**Verified feature run:** `{SUMMARY['fingerprint']}`  \n"
                 f"**Candidates:** {SUMMARY['feature_count']:,} · "
                 f"**Temporal ablation fits:** {SUMMARY['fold_tasks']:,}"))


In [ ]:
# One notebook output with interactive Plotly and a static PNG fallback for GitHub.
import base64
import io
import matplotlib.pyplot as plt

def bar_view(frame, x, y, title):
    data = frame.sort_values(y).copy()
    fig = px.bar(data, x=y, y=x, orientation="h", title=title,
                 hover_data=list(data.columns), labels={"macro_season_brier": "Mean season Brier · lower is better"}, height=max(350, 26 * len(data) + 100))
    fig.update_layout(template="plotly_white", margin=dict(l=150, r=25, t=65, b=55))
    fig.update_yaxes(autorange="reversed")
    static, ax = plt.subplots(figsize=(9.5, max(3.2, .27 * len(data) + 1.2)))
    ax.barh(data[x].astype(str), data[y], color="#147d82")
    ax.invert_yaxis()
    ax.set(xlabel="Mean season Brier · lower is better" if y == "macro_season_brier" else y.replace("_", " "), title=title)
    ax.spines[["top", "right"]].set_visible(False)
    static.tight_layout()
    buffer = io.BytesIO()
    static.savefig(buffer, format="png", dpi=140)
    plt.close(static)
    display({"application/vnd.plotly.v1+json": json.loads(fig.to_json()),
             "image/png": base64.b64encode(buffer.getvalue()).decode()}, raw=True)


## The candidate space

The existing 124 signals are supplemented by 2,944 distribution, venue, opponent-strength, trajectory and peer-profile candidates, plus 26 official coach-history and 12 conference-strength signals. Fixed windows cover the season, last seven games, last 30 days and last 60 days. Summaries cover center, spread, tails, skew and support. Rates use basketball denominators; undefined rates stay missing. Candidate counts are not independent statistical hypotheses: correlation screening deliberately removes redundant representations.

In [ ]:
registry = pd.read_csv(RESULTS / "feature_registry.csv")
assert registry.feature.is_unique and len(registry) == 3106
assert not {"y", "ID", "Season", "DayNum"}.intersection(registry.feature)
inventory = registry.groupby("family").agg(candidates=("feature", "size"), provenance=("source", "first")).reset_index()
table(inventory)
bar_view(inventory, "family", "candidates", "Candidate allocation by basketball mechanism")

## Availability is checked season by season

All current-season performance is frozen at day 132. Coach identity must be active at that cutoff; coach performance and tenure use strictly earlier seasons. Missing women’s coach records are not invented. Men’s Massey coverage is measured from legal publication dates for every modeled season, rather than assuming rankings exist only in recent years. Women’s models use the common/no-Massey schema.

In [ ]:
coverage = pd.read_csv(RESULTS / "coverage.csv")
table(coverage, {"detailed_coverage": "{:.1%}", "clean_coverage": "{:.1%}"})
display(Markdown("**Verified source availability:** `" + json.dumps(SUMMARY["sources"], sort_keys=True) + "`"))
assert not coverage.query("Gender == 'M' and Season in [2016,2017,2018,2019,2021]").ranking_teams.eq(0).any()
source_coverage = pd.read_csv(RESULTS / "source_coverage.csv")
table(source_coverage.query("Gender == 'M'")[["Season", "regular_teams", "ranking_teams", "ranking_systems", "ranking_fraction", "coach_teams", "conference_teams", "latest_ranking_day"]])
assert source_coverage.latest_ranking_day.dropna().le(132).all()


## Historical targets never cross a season boundary

Team, seed and ranking-bin encodings retain each historical season’s seed and ranking. Season Y uses outcomes strictly before Y; the entire current tournament is embargoed. Fixed priors and smoothing are not estimated from future validation seasons. Coach regular-season and tournament histories also exclude current-season outcomes. Automated mutation tests alter future labels and post-cutoff games and require identical earlier features.

In [ ]:
encoding = pd.read_csv(RESULTS / "encoding_audit.csv")
observed = encoding.loc[encoding.history_max_season.notna()]
assert (observed.history_max_season < observed.Season).all()
table(encoding.groupby(["Gender", "Season"]).agg(encoding_families=("category", "size"), history_max_season=("history_max_season", "max")).reset_index())

## Screening happens inside each training fold

There is no global feature list selected using validation outcomes. Each training-only screen audits missingness, constants, rare nonzero support, univariate association and absolute correlation. A fixed capacity of 128 limits model dimensionality; duplicate and sign-reversed duplicate signals compete for one slot. Selection, imputation and fitted models are persisted together. Correlation pruning does not prove that rejected inputs are causally irrelevant, and univariate screening can miss pure interaction effects.

In [ ]:
screening = pd.read_csv(RESULTS / "screening_summary.csv")
assert (screening.candidate_count == screening.retained_count + screening.rejected_count).all()
assert screening.retained_count.between(1, 128).all()
full = screening.query("block == 'full'")
table(full[["Gender", "Season", "model", "candidate_count", "retained_count", "rejected_count"]])
reasons = [c for c in ["all_missing", "constant", "near_constant", "redundant", "capacity", "no_training_signal", "retained"] if c in full]
table(full.groupby(["Gender", "model"])[reasons].sum().reset_index())

## Selection stability across earlier-season fits

Retention frequency measures whether a signal survives different training histories, not whether it is statistically significant. This table uses the full candidate block only. All underlying per-fit rejection reasons remain in the durable artifact archive.

In [ ]:
stability = pd.read_csv(RESULTS / "selection_stability.csv").merge(registry[["feature", "family"]], on="feature", validate="many_to_one")
table(stability.sort_values(["retention_rate", "retained_folds"], ascending=False).groupby(["Gender", "model"], sort=False).head(8))

## Does a feature family improve Brier score?

The official metric formula is game-weighted Brier, mean squared probability error. Mean-season Brier is the prespecified selection criterion and is reported separately. Whole physical tournament games are scored once; mirrored rows occur only in training. These five development seasons and the later consumed benchmark are retrospective, not an untouched holdout.

In [ ]:
scores = pd.read_csv(RESULTS / "leaderboard.csv")
table(scores.sort_values(["Gender", "macro_season_brier"]).groupby(["Gender", "model"], sort=False).head(6)[["Gender", "model", "block", "games", "brier", "macro_season_brier", "log_loss", "roc_auc"]])
# Fixed, question-driven comparison; the complete leaderboard remains in the report.
comparison_blocks = {
    "strength": "Strength baseline",
    "baseline_124": "Original bank",
    "full": "Expanded bank",
    "expanded_non_massey": "Expanded without Massey",
    "expanded_non_massey_no_coach": "Expanded without Massey/coaches",
    "expanded_non_massey_no_target": "Expanded without Massey/encoding",
    "rankings": "Strength + rankings",
    "coach_history": "Strength + coach history",
    "conference": "Strength + conference",
    "target_team": "Strength + team encoding",
    "target_seed": "Strength + seed encoding",
    "target_rank": "Strength + rank encoding",
    "dynamic": "Strength + dynamic ratings",
    "four_factors": "Strength + four factors",
}
for gender, label in [("M", "Men"), ("W", "Women")]:
    shown = scores.query("Gender == @gender and model == 'logistic'")
    shown = shown.loc[shown.block.isin(comparison_blocks)].copy()
    shown["Feature group"] = shown.block.map(comparison_blocks)
    bar_view(shown[["Feature group", "block", "brier", "macro_season_brier"]],
             "Feature group", "macro_season_brier", label + " · feature hypotheses under one logistic recipe")
display(Markdown("Focused comparison of the requested feature hypotheses. The complete block "
                 "leaderboard and histogram-boosting results remain in the recorded CSV evidence."))


## Paired ablations and uncertainty

Negative Brier differences favor the candidate. Intervals resample complete seasons and pair the same games; with only five development seasons and many comparisons they are exploratory, not corrected significance claims. The explicit `without_massey` comparison removes all 23 ranking-derived inputs together, rather than leaving trend or target-ranking derivatives behind.

In [ ]:
intervals = pd.read_csv(RESULTS / "ablation_intervals.csv")
new_families = ["distribution", "venue_profile", "opponent_profile", "trajectory", "peer_profile", "coach_history", "conference"]
selected = intervals.loc[(intervals.candidate.isin(new_families) & intervals.baseline.eq("strength")) | intervals.baseline.eq("without_massey")]
table(selected[["Gender", "model", "candidate", "baseline", "brier_delta", "ci_low", "ci_high", "season_count"]])
assert selected.season_count.eq(5).all()
controlled = intervals.loc[intervals.candidate.isin(["full", "expanded_non_massey"]) & intervals.baseline.str.startswith(("baseline_124", "expanded_non_massey"))]
table(controlled[["Gender", "model", "candidate", "baseline", "brier_delta", "ci_low", "ci_high", "season_count"]])


## Durability and interpretation

Snapshots are checkpointed by season and population. Submission features are generated in bounded 512-pair chunks; completed chunks are checksum-verified on resume. Every estimator, selection decision and prediction task has source/data/runtime fingerprints, UTC progress events and heartbeats. A changed feature definition cannot silently reuse old scores. Notebook 03 retrains against this exact feature fingerprint; notebook 05 compares matched old/new predictions. The largest candidate block is not automatically the winning model.